In [ ]:
# pip install pymysql
# pip install cryptography
# import pymysql
# conn=pymysql.connect(user='root',password='Your_Password',host='localhost')

# qur='create database emp_db'
# mycur=conn.cursor()
# mycur.execute(qur)
# mycur.close()
# conn.close()


# import pymysql
# conn=pymysql.connect(user='root',password='Your_Password',
#                              host='localhost',database='emp_db')

# qur='''create table emp(emp_id VARCHAR(40) PRIMARY KEY,emp_name VARCHAR(50),
# mob_no VARCHAR(40),dept VARCHAR(40),emp_salary VARCHAR(40))'''
# mycur=conn.cursor()
# mycur.execute(qur)
# mycur.close()
# conn.close()

In [ ]:
import config
import pymysql
from tkinter import *
from tkinter import messagebox
from tkinter import ttk

class EmployeeSystem:
    def __init__(self, win):
        self.win = win
        self.win.title("Employee Management System")
        self.win.geometry("1300x600")
        self.win.config(bg="grey")

        # Variables
        self.id1 = StringVar()
        self.n1 = StringVar()
        self.m1 = StringVar()
        self.d1 = StringVar()
        self.s1 = StringVar()
        self.var1 = StringVar()
        self.var2 = StringVar()

        self.create_widgets()

    #  Database Connection
    def connect_db(self):
    try:
        return pymysql.connect(
            host=config.DB_HOST,
            user=config.DB_USER,
            password=config.DB_PASSWORD,
            database=config.DB_NAME
        )
    except Exception as e:
        messagebox.showerror("Database Error", str(e))
        return None

    #  UI
    def create_widgets(self):

        Label(self.win, text="Employee Management System",
              bg="white", fg="black",
              font=("Times new roman", 18, 'bold'),
              width=40).place(x=350, y=20)

    # Labels & Entries
        Label(self.win, text="EMPLOYEE ID").place(x=100, y=80)
        Entry(self.win, textvariable=self.id1).place(x=250, y=80)

        Label(self.win, text="EMPLOYEE NAME").place(x=100, y=120)
        Entry(self.win, textvariable=self.n1).place(x=250, y=120)

        Label(self.win, text="MOBILE NUMBER").place(x=100, y=160)
        Entry(self.win, textvariable=self.m1).place(x=250, y=160)

        Label(self.win, text="DEPARTMENT").place(x=100, y=200)
        self.dept = ttk.Combobox(self.win, textvariable=self.d1,
                                 values=('R&D', 'Marketing', 'Production', 'HR'),
                                 state="readonly")
        self.dept.place(x=250, y=200)
        self.dept.current(0)
        self.d1.set('R&D')

        Label(self.win, text="SALARY").place(x=100, y=240)
        Entry(self.win, textvariable=self.s1).place(x=250, y=240)

    # Buttons
        Button(self.win, text="ADD", width=12, command=self.add).place(x=150, y=300)
        Button(self.win, text="SHOW", width=12, command=self.show).place(x=300, y=300)
        Button(self.win, text="CLEAR", width=12, command=self.clear).place(x=450, y=300)
        Button(self.win, text="EXIT", width=12, command=self.win.destroy).place(x=600, y=300)
        Button(self.win, text="SALARY", width=12, command=self.salary).place(x=750, y=300)

    # Delete
        Label(self.win, text="DELETE ID").place(x=100, y=350)
        Entry(self.win, textvariable=self.var1).place(x=250, y=350)
        Button(self.win, text="DELETE", command=self.delete).place(x=450, y=350)

    # Update
        Label(self.win, text="UPDATE ID").place(x=100, y=400)
        Entry(self.win, textvariable=self.var2).place(x=250, y=400)
        Button(self.win, text="SELECT", command=self.select).place(x=400, y=400)
        Button(self.win, text="UPDATE", command=self.update).place(x=500, y=400)

    # Treeview
        self.treev = ttk.Treeview(self.win,
                                  columns=("1","2","3","4","5"),
                                  show='headings')

        self.treev.place(x=700, y=80, width=550, height=400)

        self.treev.heading("1", text="Emp ID")
        self.treev.heading("2", text="Name")
        self.treev.heading("3", text="Mobile")
        self.treev.heading("4", text="Department")
        self.treev.heading("5", text="Salary")

        self.treev.column("1", width=80, anchor='center')
        self.treev.column("2", width=120, anchor='center')
        self.treev.column("3", width=120, anchor='center')
        self.treev.column("4", width=120, anchor='center')
        self.treev.column("5", width=100, anchor='center')

    #  Add
    def add(self):
        if self.id1.get() == "" or self.n1.get() == "" or self.m1.get() == "" or self.s1.get() == "":
            messagebox.showinfo("Error", "All fields are required")
            return

        if not self.m1.get().isdigit() or len(self.m1.get()) != 10:
            messagebox.showinfo("Error", "Invalid Mobile Number")
            return

        if not self.s1.get().isdigit():
            messagebox.showinfo("Error", "Salary must be numeric")
            return

        conn = self.connect_db()
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO emp VALUES (%s,%s,%s,%s,%s)",
            (self.id1.get(), self.n1.get(), self.m1.get(),
             self.d1.get(), self.s1.get())
        )

        conn.commit()
        conn.close()

        messagebox.showinfo("Success", f"Added to {self.d1.get()} department")
        self.show()
        self.clear()

    #  Show
    def show(self):
        self.treev.delete(*self.treev.get_children())

        conn = self.connect_db()
        cur = conn.cursor()
        cur.execute("SELECT * FROM emp")

        for row in cur.fetchall():
            self.treev.insert('', END, values=(
                row[0], row[1], row[2], row[3], row[4]
            ))

        conn.close()

    #  Delete
    def delete(self):
        if self.var1.get() == "":
            messagebox.showinfo("Error", "Enter ID to delete")
            return

        conn = self.connect_db()
        cur = conn.cursor()

        cur.execute("DELETE FROM emp WHERE emp_id=%s", (self.var1.get(),))
        conn.commit()
        conn.close()

        messagebox.showinfo("Success", "Deleted Successfully")
        self.show()

    #  Select
    def select(self):
        conn = self.connect_db()
        cur = conn.cursor()

        cur.execute("SELECT * FROM emp WHERE emp_id=%s", (self.var2.get(),))
        row = cur.fetchone()

        if row:
            self.id1.set(row[0])
            self.n1.set(row[1])
            self.m1.set(row[2])
            self.d1.set(row[3])
            self.s1.set(row[4])
        else:
            messagebox.showinfo("Error", "Record Not Found")

        conn.close()

    #  Update
    def update(self):
        conn = self.connect_db()
        cur = conn.cursor()

        cur.execute("""
            UPDATE emp 
            SET emp_name=%s, mob_no=%s, dept=%s, emp_salary=%s
            WHERE emp_id=%s
        """, (
            self.n1.get(),
            self.m1.get(),
            self.d1.get(),
            self.s1.get(),
            self.id1.get()
        ))

        conn.commit()
        conn.close()

        messagebox.showinfo("Success", "Updated Successfully")
        self.show()
        self.clear()

    #  Clear
    def clear(self):
        self.id1.set('')
        self.n1.set('')
        self.m1.set('')
        self.s1.set('')
        self.d1.set('R&D')
        self.var1.set('')
        self.var2.set('')
        self.treev.delete(*self.treev.get_children())

    #  Total Salary
    def salary(self):
        conn = self.connect_db()
        cur = conn.cursor()

        cur.execute("SELECT SUM(emp_salary) FROM emp")
        total = cur.fetchone()[0]

        messagebox.showinfo("Total Salary", f"Total Salary = {total}")

        conn.close()


#  Run App
if __name__ == "__main__":
    win = Tk()
    app = EmployeeSystem(win)
    win.mainloop()